# Fitbit fitness tracker analysis: Data cleaning

## 1. Overview
This notebook establishes an end-to-end data cleaning and feature engineering pipeline for the Fitbit fitness tracker dataset. Drawing from raw multi-table inputs (`dailyActivity`, `sleepDay`, `hourlyIntensity`, `hourlySteps`, and `hourlyCalories`), this script handles timestamp normalization, duplicate purging, granularity alignment, and structural left joins to prepare a pristine, analysis-ready dataset. The clean dataframes are structured for downstream exploratory data analysis (EDA). 

In [162]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from pathlib import Path

# Loading daily activity and sleep primary datasets
df_activity = pd.read_csv("../dailyActivity_merged.csv")
df_sleep = pd.read_csv("../sleepDay_merged.csv")

# Loading hourly activity, intensity, and calorie datasets
df_hourly_intensity = pd.read_csv("../hourlyIntensities_merged.csv")
df_hourly_calories = pd.read_csv("../hourlyCalories_merged.csv")
df_hourly_steps = pd.read_csv("../hourlySteps_merged.csv")

## 2. Initial data inspection
Examining individual datasets for the following to flag anomalies:
* dataset shapes
* feature data types
* missing values
* duplicate records

In [163]:
# Inspecting shapes and data types for all tables
for name, df in [('Activity', df_activity), ('Sleep', df_sleep), 
                   ('Intensity', df_hourly_intensity), ('Calories', df_hourly_calories), ('Steps', df_hourly_steps)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(df.dtypes, "\n")

# Auditing missing values across inputs
for name, df in [('Activity', df_activity), ('Sleep', df_sleep), 
                   ('Intensity', df_hourly_intensity), ('Calories', df_hourly_calories), ('Steps', df_hourly_steps)]:
    print(f"--- {name} ---")
    print(f"Missing Values:\n{df.isna().sum()}\n")

# Auditing duplicate rows across inputs
for name, df in [('Activity', df_activity), ('Sleep', df_sleep), 
                   ('Intensity', df_hourly_intensity), ('Calories', df_hourly_calories), ('Steps', df_hourly_steps)]:
    print(f"--- {name} ---")
    print(f"Duplicate Rows:\n{df.duplicated().sum()}\n")

# Inspecting exact duplicate rows in sleep data based on 'Id' and 'SleepDay' keys
duplicates = df_sleep[df_sleep.duplicated(subset=["Id", "SleepDay"], keep=False)]
print("Sample Sleep Duplicates:\n", duplicates.sort_values(["Id", "SleepDay"]))

--- Activity ---
Shape: (940, 15)
Id                            int64
ActivityDate                    str
TotalSteps                    int64
TotalDistance               float64
TrackerDistance             float64
LoggedActivitiesDistance    float64
VeryActiveDistance          float64
ModeratelyActiveDistance    float64
LightActiveDistance         float64
SedentaryActiveDistance     float64
VeryActiveMinutes             int64
FairlyActiveMinutes           int64
LightlyActiveMinutes          int64
SedentaryMinutes              int64
Calories                      int64
dtype: object 

--- Sleep ---
Shape: (413, 5)
Id                    int64
SleepDay                str
TotalSleepRecords     int64
TotalMinutesAsleep    int64
TotalTimeInBed        int64
dtype: object 

--- Intensity ---
Shape: (22099, 4)
Id                    int64
ActivityHour            str
TotalIntensity        int64
AverageIntensity    float64
dtype: object 

--- Calories ---
Shape: (22099, 3)
Id              int64
Act

## 3. "Daily data" - Cleaning & Preprocessing
* **Timestamp normalization:** Converting string dates into standardized `datetime` format.
* **Handling duplicates:** Identifying and dropping exact-row duplicates within the sleep tracking dataset.

In [ ]:
df_activity['Date'] = pd.to_datetime(df_activity['ActivityDate'], format="%m/%d/%Y")
df_sleep['Date'] = pd.to_datetime(df_sleep['SleepDay'], format="%m/%d/%Y %I:%M:%S %p").dt.normalize()

print(
    "Are both dates the same type?",
    df_activity["ActivityDate"].dtype == df_sleep["SleepDay"].dtype
)

# Removing exact-row duplicate records from sleep dataset after insection
## Reasoning - duplicates are highly unlikely to be different sleep session due to identical durations
df_sleep.drop_duplicates(subset=["Id", "Date"], keep='first', inplace=True)
print("Duplicates in Sleep after cleaning:", df_sleep.duplicated().sum())

# Verifying if all sleep IDs are a strict subset of the activity dataset
all_sleep_in_activity = df_sleep['Id'].isin(df_activity['Id']).all()
print("Are all sleep IDs are present in activity data?", all_sleep_in_activity)

# Verifying unique user counts
print("Unique IDs in Activity:", df_activity['Id'].nunique())
print("Unique IDs in Sleep:", df_sleep['Id'].nunique())

print(df_sleep.head())


Are both dates the same type? True
Duplicates in Sleep after cleaning: 0
Are all sleep IDs are present in activity data? True
Unique IDs in Activity: 33
Unique IDs in Sleep: 24
           Id               SleepDay  TotalSleepRecords  TotalMinutesAsleep  \
0  1503960366  4/12/2016 12:00:00 AM                  1                 327   
1  1503960366  4/13/2016 12:00:00 AM                  2                 384   
2  1503960366  4/15/2016 12:00:00 AM                  1                 412   
3  1503960366  4/16/2016 12:00:00 AM                  2                 340   
4  1503960366  4/17/2016 12:00:00 AM                  1                 700   

   TotalTimeInBed       Date  
0             346 2016-04-12  
1             407 2016-04-13  
2             442 2016-04-15  
3             367 2016-04-16  
4             712 2016-04-17  


## 4. "Daily data" - Feature engineering
* extracting date/time attributes (`Day_of_the_week`, `Is_weekend`)
* deriving clinical/behavioral sleep features (`HoursAsleep`, `HoursInBed`, and `SleepEfficiency`)
* engineering features, `TotalTrackerMinites` and `ValidWearDay`, to classify device wear/non wear days

In [ ]:
# Extracting temporal features
df_activity['Day_of_the_week'] = df_activity['Date'].dt.day_name() 
df_activity['Is_Weekend'] = df_activity['Day_of_the_week'].isin(['Saturday', 'Sunday'])

# Engineering sleep metrics, avoiding cases of zero time in bed to prevent division by zero
df_sleep['HoursAsleep'] = df_sleep['TotalMinutesAsleep'] / 60
df_sleep['HoursInBed'] = df_sleep['TotalTimeInBed'] / 60
df_sleep['SleepEfficiency'] = np.where(
    df_sleep['TotalTimeInBed'] > 0,
    df_sleep['TotalMinutesAsleep'] / df_sleep['TotalTimeInBed'],
    np.nan
)
# Optional check for physical plausibility
df_sleep['SleepEfficiency'] = df_sleep['SleepEfficiency'].clip(upper=1.0)

# Total active + sedentary minutes recorded in the 24-hr day (1440 min max)
## to classify days into valid wear vs non-wear days for activity tracking
df_activity['TotalTrackerMinutes'] = (
    df_activity['VeryActiveMinutes'] +
    df_activity['FairlyActiveMinutes'] +
    df_activity['LightlyActiveMinutes'] +
    df_activity['SedentaryMinutes']
)

# To Filter out/flag non-wear days 
## Assumptions: Tracker worn for >= 10 hours (600 mins) and steps > 0
df_activity['ValidWearDay'] = (df_activity['TotalTrackerMinutes'] >= 600) & (df_activity['TotalSteps'] > 0)


print("Activity DataFrame Info:")
print(df_activity.head())
print("\n")
print("Sleep DataFrame Info:")
print(df_sleep.head())

print("Missing vals",df_sleep.isna().sum())



Activity DataFrame Info:
           Id ActivityDate  TotalSteps  TotalDistance  TrackerDistance  \
0  1503960366    4/12/2016       13162           8.50             8.50   
1  1503960366    4/13/2016       10735           6.97             6.97   
2  1503960366    4/14/2016       10460           6.74             6.74   
3  1503960366    4/15/2016        9762           6.28             6.28   
4  1503960366    4/16/2016       12669           8.16             8.16   

   LoggedActivitiesDistance  VeryActiveDistance  ModeratelyActiveDistance  \
0                       0.0                1.88                      0.55   
1                       0.0                1.57                      0.69   
2                       0.0                2.44                      0.40   
3                       0.0                2.14                      1.26   
4                       0.0                2.71                      0.41   

   LightActiveDistance  SedentaryActiveDistance  VeryActiveMinutes 

## 5. "Daily" and "Hourly datasets" - temporal overlap audit
Checking for timeframe overlap for succesful merging.

In [166]:
# Checking date range overlaps for daily/sleep data 
print("Activity Date Range:", df_activity["Date"].min(), "to", df_activity["Date"].max())
print("Sleep Date Range:", df_sleep["Date"].min(), "to", df_sleep["Date"].max())

# Overlap outcome
activity_dates = set(df_activity['Date'].dt.date)
sleep_dates = set(df_sleep['Date'].dt.date)

if activity_dates == sleep_dates:
    print("Daily and sleep date sets match perfectly.")
elif activity_dates.isdisjoint(sleep_dates):
    print("Warning: No temporal overlap between activity and sleep datasets.")
else:
    print("Partial overlap: Sleep dates intersect with activity dates.")

# Checking timestamp equality for hourly data
set_intensity = set(df_hourly_intensity['ActivityHour'])
set_calories = set(df_hourly_calories['ActivityHour'])
set_steps = set(df_hourly_steps['ActivityHour'])

# Overlap outcome
if set_intensity == set_calories == set_steps:
    print("Hourly timestamp sets match perfectly.")
else:
    print("Warning: Discrepancy found in hourly timestamp sets.")

Activity Date Range: 2016-04-12 00:00:00 to 2016-05-12 00:00:00
Sleep Date Range: 2016-04-12 00:00:00 to 2016-05-12 00:00:00
Daily and sleep date sets match perfectly.
Hourly timestamp sets match perfectly.


In [167]:
df_activity.head()
df_sleep.head()


,Id,SleepDay,TotalSleepRecords,TotalMinutesAsleep,TotalTimeInBed,Date,HoursAsleep,HoursInBed,SleepEfficiency
0,1503960366,4/12/2016 12:00:00 AM,1,327,346,2016-04-12,5.450000,5.766667,0.945087
1,1503960366,4/13/2016 12:00:00 AM,2,384,407,2016-04-13,6.400000,6.783333,0.943489
2,1503960366,4/15/2016 12:00:00 AM,1,412,442,2016-04-15,6.866667,7.366667,0.932127
3,1503960366,4/16/2016 12:00:00 AM,2,340,367,2016-04-16,5.666667,6.116667,0.926431
4,1503960366,4/17/2016 12:00:00 AM,1,700,712,2016-04-17,11.666667,11.866667,0.983146


## 6. "Daily dataset" - Merging
Executing a left join on user `Id` and date (`ActivityDate` matching `SleepDay`) to preserve all 33 user activity profiles while maintaining structural alignment with the 24-user sleep subset.

In [168]:
# Merging activity and sleep tables via left join
df_merged_daily = pd.merge(
    df_activity, df_sleep, 
    on=['Id', 'Date'], 
    how='left'
)

# Cleaning up redundant key columns and sort chronologically
#df_merged_daily = df_merged_daily.drop(columns=['Date'])
df_merged_daily = df_merged_daily.sort_values(['Id', 'Date']).reset_index(drop=True)

## 7. "Hourly datasets" - Preprocessing & Consolidation
* **Timestamp normalization:** Converting string dates into standardized `datetime` format.
* **Feature engineering:** Extracting date/hour attributes (`Date`, `Hour`, `DayOfWeek`, `IsWeekend`).
* **Merging hourly datasets:** Left joining on user `Id` and `ActivityHour`.

In [169]:
# Standardize timestamp
for df in [df_hourly_intensity, df_hourly_steps, df_hourly_calories]:
    df['ActivityHour'] = pd.to_datetime(df['ActivityHour'], format='%m/%d/%Y %I:%M:%S %p')

# Verify if all hourly files share identical ID sets
identical_hourly_ids = (set(df_hourly_intensity['Id']) == set(df_hourly_calories['Id']) == set(df_hourly_steps['Id']))
print("Identical IDs across all hourly datasets:", identical_hourly_ids)

# Merge cleanly on keys only
df_merged_hourly = (
    df_hourly_intensity
    .merge(df_hourly_steps[['Id', 'ActivityHour', 'StepTotal']], on=['Id', 'ActivityHour'], how='left')
    .merge(df_hourly_calories[['Id', 'ActivityHour', 'Calories']], on=['Id', 'ActivityHour'], how='left')
)

# Engineer temporal attributes once on the merged frame
df_merged_hourly['Date'] = df_merged_hourly['ActivityHour'].dt.date
df_merged_hourly['Hour'] = df_merged_hourly['ActivityHour'].dt.hour
df_merged_hourly['DayOfWeek'] = df_merged_hourly['ActivityHour'].dt.day_name()
df_merged_hourly['IsWeekend'] = df_merged_hourly['ActivityHour'].dt.dayofweek >= 5



Identical IDs across all hourly datasets: True


## 7. Final checks & export
* Performing a final validation pass on shapes and missing values.
* Exporting the cleaned datasets to CSV for future EDA.

In [170]:
# Final validation checks
print("Daily Clean Shape:", df_merged_daily.shape)
print("Hourly Clean Shape:", df_merged_hourly.shape)
print("\nDaily Missing Values:\n", df_merged_daily.isna().sum())
print("\nHourly Missing Values:\n", df_merged_hourly.isna().sum())

# Exporting clean files for the EDA workflow
df_merged_daily.to_csv("daily_merged_clean.csv", index=False)
df_merged_hourly.to_csv("hourly_merged_clean.csv", index=False)
print("\nPipeline execution complete: Cleaned datasets exported successfully.")

Daily Clean Shape: (940, 27)
Hourly Clean Shape: (22099, 10)

Daily Missing Values:
 Id                            0
ActivityDate                  0
TotalSteps                    0
TotalDistance                 0
TrackerDistance               0
LoggedActivitiesDistance      0
VeryActiveDistance            0
ModeratelyActiveDistance      0
LightActiveDistance           0
SedentaryActiveDistance       0
VeryActiveMinutes             0
FairlyActiveMinutes           0
LightlyActiveMinutes          0
SedentaryMinutes              0
Calories                      0
Date                          0
Day_of_the_week               0
Is_Weekend                    0
TotalTrackerMinutes           0
ValidWearDay                  0
SleepDay                    530
TotalSleepRecords           530
TotalMinutesAsleep          530
TotalTimeInBed              530
HoursAsleep                 530
HoursInBed                  530
SleepEfficiency             530
dtype: int64

Hourly Missing Values:
 Id           